# 09 · Cronolecto: cómo nombra la prensa (Entrega 3, S2)

El 08 mide **qué** temas cubre la prensa colombiana; este notebook mide **cómo** los nombra. Responde dos observaciones del profesor sobre la entrega 2:

* **(a)** exploración cronolectal completa: expresiones distintas que refieren a lo mismo y se reemplazan con el tiempo (secciones 3, 4 y 8);
* **(b)** rareza con frecuencia por millón (pmw) y escala Zipf, con sus implicaciones demostradas en el corpus (sección 2).

**Controles que atraviesan todo el notebook:**

| confusión | control |
|---|---|
| fuente confundida con tiempo (años 80 ≈ Semana) | serie principal El Tiempo 1990–2011; Semana 1982–2011 como contraste; Dinero solo donde hay N |
| formato de El Tiempo (breves, titulares, cambio de archivo en 2006) | base sin breves ni cartas; sensibilidad con todo; todo cruce o cambio se marca si cae a ±1 de 2006 |
| composición temática | cuotas estandarizadas por tópico dominante del 08 |
| conteos bajos | NaN explícito por debajo del mínimo; ninguna función devuelve listas de relleno |

**Celdas costosas** (marcadas con ⏱): matriz de formas (~2 min), conteo de variantes (~1–2 min), cuotas con bootstrap (~2–3 min), word2vec base + con breves (~20–30 min la primera vez; luego se cargan del caché), vecindario local (~2 min). **Tiempo medido en local: ~6 min la primera corrida de las secciones 0–9; la sección 4b agrega ~4 min (40 modelos más).** `FAST = True` usa el parquet de 6.000 documentos (~3–5 min).

## 0. Reglas congeladas

Todo lo que decide resultados se declara aquí, **antes de ver cualquier salida**: configuración, conjuntos de variantes, semillas semánticas y términos de paz. El hash se guarda en `results/09/congelado.json` en la primera corrida; en las siguientes, si algo de esto cambió, la celda falla. Lo que se descubra después va en `EXPLORATORIO` (sección 8), que no entra al hash.

In [ ]:
import json, re, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import utils, agenda
import cronolecto as cr
from agenda import ET, SEMANA, DINERO, CORTO

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 200)

FAST = False
cfg = cr.ConfigCrono.rapida() if FAST else cr.ConfigCrono()
RES = utils.results_dir() / ("09_fast" if FAST else "09")
RES.mkdir(parents=True, exist_ok=True)
CACHE = RES / "cache"
CACHE.mkdir(exist_ok=True)
R08 = utils.results_dir() / "08"
TIEMPOS, T0 = {}, time.time()


def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(RES / f"{nombre}.png", dpi=110)
    plt.show()


print("FAST =", FAST, "| resultados en", RES)

**Conjuntos de variantes (congelados).** `tipo`:
* `reemplazo`: mismo referente, otra forma (ortografía, toponimia, préstamo, gentilicio, denominación de grupos);
* `referente_cercano`: no son sinónimos estrictos (VIH es el virus, sida la enfermedad);
* `marco`: referente cercano con valoración distinta (conflicto, paz). **No se leen como sinónimos.**

`cruce` es la hipótesis declarada (vieja → nueva); `None` = sin hipótesis de reemplazo: solo cuotas y detector de cambio.

Advertencias declaradas antes de ver datos:
* `solo` incluye el adjetivo ("un solo candidato"): la cuota de `sólo` es una **cota inferior** del uso con tilde en el adverbio.
* `éste/este` no se mide como cuota (el determinante domina); se reporta el pmw de las formas tildadas.
* `pc` puede ser Partido Conservador o Comunista; `canje` puede ser canje de deuda; `retenido` puede ser retención policial: se verifica con contextos.
* `tokens_forma` no tiene cortes de puntuación: una forma de varias palabras podría cruzar una coma (raro; se acepta).

In [ ]:
def _flex(*bases):  # masculino/femenino, singular/plural de adjetivos en -o
    out = []
    for b in bases:
        out += [b, b[:-1] + "a", b + "s", b[:-1] + "as"]
    return out


VARIANTES = {
    # --- ortográficas ---
    "solo_tilde": {
        "categoria": "ortografica",
        "tipo": "reemplazo",
        "cruce": ("sólo", "solo"),
        "variantes": {"sólo": ["sólo"], "solo": ["solo"]},
    },
    "guion": {
        "categoria": "ortografica",
        "tipo": "reemplazo",
        "cruce": ("guión", "guion"),
        "variantes": {"guión": ["guión"], "guion": ["guion"]},
    },
    "periodo": {
        "categoria": "ortografica",
        "tipo": "reemplazo",
        "cruce": ("período", "periodo"),
        "variantes": {
            "período": ["período", "períodos"],
            "periodo": ["periodo", "periodos"],
        },
    },
    "pekin": {
        "categoria": "ortografica",
        "tipo": "reemplazo",
        "cruce": ("pekín", "beijing"),
        "variantes": {"pekín": ["pekín", "pekin"], "beijing": ["beijing"]},
    },
    "irak": {
        "categoria": "ortografica",
        "tipo": "reemplazo",
        "cruce": ("irak", "iraq"),
        "variantes": {"irak": ["irak"], "iraq": ["iraq"]},
    },
    # --- toponimia e institucional ---
    "santafe": {
        "categoria": "toponimia",
        "tipo": "reemplazo",
        "cruce": None,
        "variantes": {
            "santafé de bogotá": [
                "santafé de bogotá",
                "santa fe de bogotá",
                "santafe de bogotá",
                "santa fé de bogotá",
                "santafé de bogota",
                "santa fe de bogota",
            ],
            "bogotá": ["bogotá", "bogota"],
        },
    },
    "distrito": {
        "categoria": "toponimia",
        "tipo": "reemplazo",
        "cruce": ("distrito especial", "distrito capital"),
        "variantes": {
            "distrito especial": ["distrito especial"],
            "distrito capital": ["distrito capital"],
        },
    },
    "upac_uvr": {
        "categoria": "institucional",
        "tipo": "reemplazo",
        "cruce": ("upac", "uvr"),
        "variantes": {"upac": ["upac"], "uvr": ["uvr"]},
    },
    # --- préstamo y tecnología ---
    "computador": {
        "categoria": "tecnologia",
        "tipo": "reemplazo",
        "cruce": None,
        "variantes": {
            "computador": ["computador", "computadores"],
            "computadora": ["computadora", "computadoras"],
            "ordenador": ["ordenador", "ordenadores"],
            "pc": ["pc", "pcs"],
        },
    },
    "celular": {
        "categoria": "tecnologia",
        "tipo": "reemplazo",
        "cruce": None,
        "variantes": {
            "celular": [
                "celular",
                "celulares",
                "teléfono celular",
                "teléfonos celulares",
                "telefonía celular",
            ],
            "móvil": ["teléfono móvil", "teléfonos móviles", "telefonía móvil"],
        },
    },
    "correo": {
        "categoria": "tecnologia",
        "tipo": "reemplazo",
        "cruce": None,
        "variantes": {
            "correo electrónico": [
                "correo electrónico",
                "correos electrónicos",
                "correo electronico",
            ],
            "e-mail": ["e-mail", "e mail", "e-mails", "e mails"],
            "email": ["email", "emails"],
        },
    },
    # --- gentilicio ---
    "gentilicio_eeuu": {
        "categoria": "gentilicio",
        "tipo": "reemplazo",
        "cruce": ("norteamericano", "estadounidense"),
        "variantes": {
            "norteamericano": _flex("norteamericano"),
            "estadounidense": ["estadounidense", "estadounidenses"],
        },
    },
    # --- salud ---
    "sida_vih": {
        "categoria": "salud",
        "tipo": "referente_cercano",
        "cruce": ("sida", "vih"),
        "variantes": {"sida": ["sida"], "vih": ["vih"]},
    },
    # --- denominación de grupos ---
    "tercera_edad": {
        "categoria": "grupos",
        "tipo": "reemplazo",
        "cruce": ("tercera edad", "adulto mayor"),
        "variantes": {
            "tercera edad": ["tercera edad"],
            "adulto mayor": [
                "adulto mayor",
                "adultos mayores",
                "adulta mayor",
                "adultas mayores",
            ],
        },
    },
    "discapacidad": {
        "categoria": "grupos",
        "tipo": "reemplazo",
        "cruce": ("minusválido", "discapacitado"),
        "variantes": {
            "minusválido": _flex("minusválido") + ["minusvalía"],
            "discapacitado": _flex("discapacitado"),
            "persona con discapacidad": [
                "persona con discapacidad",
                "personas con discapacidad",
                "población con discapacidad",
                "situación de discapacidad",
            ],
        },
    },
    "afro": {
        "categoria": "grupos",
        "tipo": "reemplazo",
        "cruce": ("negritudes", "afrocolombiano"),
        "variantes": {
            "negritudes": [
                "negritudes",
                "negritud",
                "comunidades negras",
                "comunidad negra",
                "población negra",
            ],
            "afrocolombiano": _flex("afrocolombiano"),
            "afrodescendiente": ["afrodescendiente", "afrodescendientes"],
        },
    },
    "indigena": {
        "categoria": "grupos",
        "tipo": "reemplazo",
        "cruce": ("indio", "indígena"),
        "variantes": {
            "indio": ["indio", "indios"],  # "india(s)" se excluye: país
            "indígena": ["indígena", "indígenas"],
            "aborigen": ["aborigen", "aborígenes"],
        },
    },
    "diversidad_sexual": {
        "categoria": "grupos",
        "tipo": "reemplazo",
        "cruce": None,
        "variantes": {
            "homosexual": ["homosexual", "homosexuales", "homosexualidad"],
            "gay": ["gay", "gays"],
            "lgbt": ["lgbt", "lgbti", "lgtb", "lgbtiq", "lgbtq"],
        },
    },
    # --- conflicto: marco o eufemismo, NO sinónimos ---
    "insurgencia": {
        "categoria": "conflicto",
        "tipo": "marco",
        "cruce": None,
        "variantes": {
            "subversión": ["subversión"] + _flex("subversivo"),
            "insurgencia": ["insurgencia", "insurgente", "insurgentes"],
            "terroristas": ["terrorista", "terroristas"],
            "grupos armados ilegales": [
                "grupo armado ilegal",
                "grupos armados ilegales",
                "grupo armado al margen de la ley",
                "grupos armados al margen de la ley",
            ],
        },
    },
    "paramilitares": {
        "categoria": "conflicto",
        "tipo": "marco",
        "cruce": None,
        "variantes": {
            "autodefensas": ["autodefensa", "autodefensas"],
            "paramilitares": ["paramilitar", "paramilitares", "paramilitarismo"],
            "bacrim": [
                "bacrim",
                "banda criminal",
                "bandas criminales",
                "bandas emergentes",
            ],
        },
    },
    "reinsertados": {
        "categoria": "conflicto",
        "tipo": "marco",
        "cruce": ("reinsertado", "desmovilizado"),
        "variantes": {
            "reinsertado": _flex("reinsertado") + ["reinserción"],
            "desmovilizado": _flex("desmovilizado") + ["desmovilización"],
        },
    },
    "secuestro": {
        "categoria": "conflicto",
        "tipo": "marco",
        "cruce": None,
        "variantes": {
            "secuestrado": _flex("secuestrado") + ["secuestro", "secuestros"],
            "retenido": _flex("retenido"),
        },
    },  # "retención" sola: retención en la fuente
    "falsos_positivos": {
        "categoria": "conflicto",
        "tipo": "marco",
        "cruce": None,
        "variantes": {
            "ejecuciones extrajudiciales": [
                "ejecución extrajudicial",
                "ejecuciones extrajudiciales",
            ],
            "falsos positivos": ["falso positivo", "falsos positivos"],
        },
    },
    "conflicto_marco": {
        "categoria": "conflicto",
        "tipo": "marco",
        "cruce": None,
        "variantes": {
            "conflicto armado": ["conflicto armado", "conflicto armado interno"],
            "amenaza terrorista": ["amenaza terrorista"],
        },
    },
    "canje": {
        "categoria": "conflicto",
        "tipo": "marco",
        "cruce": None,
        "variantes": {
            "canje": ["canje"],
            "acuerdo humanitario": ["acuerdo humanitario"],
            "intercambio humanitario": ["intercambio humanitario"],
        },
    },
    # --- paz (puente con la sección 7) ---
    "paz_denominacion": {
        "categoria": "paz",
        "tipo": "marco",
        "cruce": None,
        "variantes": {
            "proceso de paz": ["proceso de paz", "procesos de paz"],
            "diálogos de paz": ["diálogo de paz", "diálogos de paz"],
            "negociaciones de paz": ["negociación de paz", "negociaciones de paz"],
            "conversaciones de paz": ["conversaciones de paz"],
            "acuerdo de paz": ["acuerdo de paz", "acuerdos de paz"],
        },
    },
}

ESTE_TILDE = ["éste", "ésta", "éstos", "éstas"]

# Semillas de cambio semántico (lemas de lemas_contenido, mismo flujo que word2vec)
SEMILLAS_SEM = [
    "paz",
    "violencia",
    "guerrilla",
    "víctima",
    "desplazado",
    "terrorismo",
    "seguridad",
    "corrupción",
    "red",
    "celular",
    "verde",
    "ambiente",
    "diálogo",
    "negociación",
    "acuerdo_paz",
]  # paz léxica (sección 7)

# Términos de paz, contados de forma independiente (pueden solaparse)
TERMINOS_PAZ = {
    "paz": ["paz"],
    "proceso de paz": ["proceso de paz", "procesos de paz"],
    "diálogo": ["diálogo", "diálogos", "dialogo", "dialogos"],
    "negociación": ["negociación", "negociaciones", "negociacion"],
    "acuerdo de paz": ["acuerdo de paz", "acuerdos de paz"],
    "zona de distensión": [
        "zona de distensión",
        "zona de despeje",
        "zona desmilitarizada",
    ],
    "acuerdo humanitario": ["acuerdo humanitario", "intercambio humanitario"],
    "desmovilización": ["desmovilización", "desmovilizaciones"],
    "reconciliación": ["reconciliación"],
    "posconflicto": ["posconflicto", "postconflicto"],
}

SERIES_DECL = {
    "eltiempo_base": "El Tiempo 1990–2011, sin breves ni cartas (serie principal)",
    "eltiempo_todos": "El Tiempo 1990–2011, todo (sensibilidad con breves)",
    "semana_base": "Semana, sin breves ni cartas (contraste largo)",
    "dinero_base": "Dinero, sin breves ni cartas (solo donde hay N)",
}

In [ ]:
DECLARADO = {
    "variantes": VARIANTES,
    "este_tilde": ESTE_TILDE,
    "semillas_sem": SEMILLAS_SEM,
    "terminos_paz": TERMINOS_PAZ,
    "series": SERIES_DECL,
}
HASH = cr.hash_congelado(cfg, DECLARADO)
print(cr.congelar(RES / "congelado.json", HASH, {"config": cfg.a_dict(), **DECLARADO}))
for c, d in VARIANTES.items():
    cr.compilar(
        d["variantes"]
    )  # falla si una forma está en dos variantes del mismo conjunto
print(
    len(VARIANTES),
    "conjuntos;",
    sum(len(d["variantes"]) for d in VARIANTES.values()),
    "variantes",
)

## 1. Carga y base

Columnas del parquet S0 v3.0.3 (no se vuelve a correr spaCy) y θ del 08 por `id`. La base es **sin breves ni cartas**, como el ajuste del 08.

In [ ]:
COLS = [
    "id",
    "fuente",
    "fecha",
    "año",
    "mes",
    "es_breve",
    "es_carta",
    "tokens_forma",
    "lemas_contenido",
    "pos_contenido",
    "cortes_contenido",
    "entidades",
    "n_palabras_cuerpo",
]
with cr.cronometro(TIEMPOS, "carga"):
    dfp = utils.load_corpus_procesado(fast=FAST, columns=COLS)
dfp = dfp[(dfp["año"] >= cfg.año_min) & (dfp["año"] <= cfg.año_max)].reset_index(
    drop=True
)
th = cr.cargar_theta(R08 / "theta_documentos.parquet")
dfp = dfp.merge(th, on="id", how="left")
assert dfp["id"].is_unique

nombres = pd.read_csv(R08 / "nombres_topicos.csv", keep_default_na=False)
NOMBRE = dict(zip(nombres["topico"], nombres["nombre"]))
RESIDUALES = set(
    nombres.loc[
        nombres["residual"].astype(str).str.lower().isin(["true", "1"]), "topico"
    ]
)

fuente = dfp["fuente"].values
año = dfp["año"].values.astype(int)
ids = dfp["id"].values
base = ~dfp["es_breve"].values & ~dfp["es_carta"].values
L_forma = dfp["tokens_forma"].map(len).values.astype(np.int64)
L_cont = dfp["lemas_contenido"].map(len).values.astype(np.int64)
per = cr.periodo_de(año, cfg.periodos)
SERIES = {
    "eltiempo_base": (fuente == ET) & base & (año >= cfg.et_desde),
    "eltiempo_todos": (fuente == ET) & (año >= cfg.et_desde),
    "semana_base": (fuente == SEMANA) & base,
    "dinero_base": (fuente == DINERO) & base,
}
print(
    f"{len(dfp):,} docs | con θ del 08: {dfp['topico'].notna().mean():.1%} | residuales del 08: {RESIDUALES}"
)
print(
    "θ dominante (cuantiles):",
    dfp["theta_dom"].quantile([0.1, 0.25, 0.5, 0.75]).round(2).to_dict(),
)
print({k: int(v.sum()) for k, v in SERIES.items()})
print(pd.crosstab(dfp["año"], dfp["fuente"].map(CORTO)).T.to_string())

## 2. Frecuencia y rareza (observación b)

**pmw** = apariciones / tokens del grupo × 10⁶; **Zipf** = log₁₀(pmw) + 3 (van Heuven et al. 2014). Bandas **declaradas**: Zipf < 3 raro, 3–4 medio, ≥ 4 frecuente. Unidad: **formas** de `tokens_forma` (con stopwords), por (fuente, año), base sin breves ni cartas. El Zipf de una palabra ausente no existe (NaN); van Heuven propone el Zipf de Laplace, log₁₀((c+1)/(N+V)·10⁹), que también se exporta.

Cada implicación se demuestra con el corpus en 2.1–2.6.

In [ ]:
# ⏱ ~2 min: matriz documento × forma
with cr.cronometro(TIEMPOS, "matriz_formas"):
    Xf, vf = cr.matriz_conteos(dfp["tokens_forma"], min_df=cfg.min_df_formas)
    ixf = {w: i for i, w in enumerate(vf)}
    tot_f = np.asarray(Xf.sum(0)).ravel()
    etq_base = [(CORTO[f], int(a)) if b else None for f, a, b in zip(fuente, año, base)]
    ok_terms = np.where(tot_f >= cfg.min_total_export)[0]
    PZ = cr.pmw_zipf(Xf, vf, L_forma, etq_base, cfg, ok_terms)
PZ.to_parquet(RES / "pmw_zipf.parquet", index=False)
SW = utils.cargar_stopwords()
es_contenido = lambda w: (
    isinstance(w, str) and w.isalpha() and len(w) >= 3 and w not in SW
)
print(
    f"{Xf.shape[0]:,} docs × {Xf.shape[1]:,} formas (min_df={cfg.min_df_formas}) | exportadas {len(ok_terms):,} formas con ≥{cfg.min_total_export} apariciones"
)
print(
    PZ[(PZ.fuente == "eltiempo") & PZ.termino.map(es_contenido)]
    .groupby("año")["banda"]
    .value_counts(normalize=True)
    .unstack()
    .round(3)
    .head(25)
)

### 2.1 El piso del Zipf depende del tamaño del año

Con N tokens, una sola aparición ya vale Zipf = log₁₀(10⁶/N) + 3. Si ese piso es ≥ 3, la banda **rara no se puede medir** en ese grupo: toda palabra presente parece "media" o "frecuente". Además, en grupos chicos el pmw de una palabra media salta entre 0 y valores altos según qué documentos entraron a la muestra.

In [ ]:
NG = pd.DataFrame(
    {"fuente": [CORTO[f] for f in fuente], "año": año, "tokens": L_forma, "base": base}
)
NG = (
    NG[NG.base]
    .groupby(["fuente", "año"])
    .agg(docs=("tokens", "size"), tokens=("tokens", "sum"))
    .reset_index()
)
NG["piso_zipf"] = cr.piso_zipf(NG["tokens"])
NG["rara_medible"] = NG["piso_zipf"] < cfg.bandas[0]
NG["grupo_chico"] = NG["tokens"] < cfg.min_tokens_grupo
print(NG.sort_values("tokens").head(12).round(2).to_string(index=False))
print(
    "\nGrupos donde la banda rara NO es medible:",
    NG.loc[~NG.rara_medible, ["fuente", "año"]]
    .astype(str)
    .agg(" ".join, axis=1)
    .tolist(),
)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for f, g in NG.groupby("fuente"):
    ax[0].plot(g["año"], g["piso_zipf"], marker="o", ms=3, label=f)
ax[0].axhline(cfg.bandas[0], ls="--", c="k", lw=0.8)
ax[0].set_title("Zipf de una sola aparición (piso medible)")
ax[0].legend()

# Submuestreo: un año grande de El Tiempo reducido al tamaño de los años chicos
y_ref = int(NG[NG.fuente == "eltiempo"].sort_values("tokens").iloc[-1]["año"])
idx_ref = np.where(SERIES["eltiempo_base"] & (año == y_ref))[0]
Xr = Xf[idx_ref]
Lr = L_forma[idx_ref]
nz = np.asarray(Xr.sum(0)).ravel()
zr_full = cr.zipf(nz / Lr.sum() * 1e6)
cand = [j for j in np.where((zr_full >= 3) & (zr_full < 3.5))[0] if es_contenido(vf[j])]
rg = np.random.default_rng(cfg.seed)
medias = np.asarray(
    rg.choice(cand, size=min(200, len(cand)), replace=False) if cand else [], dtype=int
)
filas = []
for n_docs in [25, 50, 100, 200, 500, 1000, len(idx_ref)] if len(medias) else []:
    n_docs = min(n_docs, len(idx_ref))
    for r in range(30):
        s = rg.choice(len(idx_ref), n_docs, replace=False)
        c = np.asarray(Xr[s][:, medias].sum(0)).ravel()
        N = Lr[s].sum()
        z = cr.zipf(c / N * 1e6)
        filas.append(
            {
                "docs": n_docs,
                "tokens": N,
                "frac_ausente": np.mean(c == 0),
                "frac_sobre_4": np.mean(z >= 4),
                "sd_zipf": np.nanstd(z - zr_full[medias]),
            }
        )
SUB = (
    pd.DataFrame(
        filas, columns=["docs", "tokens", "frac_ausente", "frac_sobre_4", "sd_zipf"]
    )
    .groupby("docs")
    .mean()
    .round(3)
)
if SUB.empty:
    print(
        "Sin palabras de contenido con Zipf 3–3,5 en el año de referencia: demostración omitida (explícito)."
    )
print(
    f"\nEl Tiempo {y_ref}: {len(medias)} palabras de contenido con Zipf 3–3,5 en el año completo, medidas en submuestras"
)
print(SUB.to_string())
ax[1].plot(SUB.index, SUB["frac_ausente"], marker="o", label="ausente (Zipf no existe)")
ax[1].plot(
    SUB.index, SUB["frac_sobre_4"], marker="o", label="parece frecuente (Zipf ≥ 4)"
)
ax[1].set_xscale("log")
ax[1].set_xlabel("docs en la muestra")
ax[1].legend()
ax[1].set_title("Palabras medias en muestras chicas")
guardar(fig, "fig_piso_zipf")

### 2.2 Inestabilidad con conteos bajos: IC Poisson frente a bootstrap por documento

El IC de Poisson supone apariciones independientes. En prensa las palabras llegan en ráfagas (un documento sobre un tema la repite), así que el IC real (bootstrap de Poisson por documento) es más ancho, y más cuanto más concentrada está la palabra (DP alto).

**Regla declarada del conteo mínimo:** el menor bin de conteo cuya mediana de semiancho relativo del IC bootstrap es ≤ 0,5.

In [ ]:
def demo_ic(serie, y, nombre):
    idx = np.where(SERIES[serie] & (año == y))[0]
    if len(idx) == 0:
        return None
    Xg, Lg = Xf[idx], L_forma[idx]
    n_t = np.asarray(Xg.sum(0)).ravel()
    pres = np.where(n_t > 0)[0]
    bins = pd.cut(
        n_t[pres],
        [1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 10**9],
        right=False,
        labels=False,
    )
    rg = np.random.default_rng(cfg.seed)
    por_bin = max(1, cfg.n_terminos_demo // 10)
    sel = np.concatenate(
        [
            rg.choice(pres[bins == b], min(por_bin, (bins == b).sum()), replace=False)
            for b in np.unique(bins[~np.isnan(bins)])
        ]
    )
    lo, hi, _ = cr.bootstrap_tasa(Xg[:, sel], Lg, cfg.b_boot, cfg.seed)
    ar = cr.ancho_relativo(n_t[sel], Lg.sum(), lo, hi)
    ar["termino"] = vf[sel]
    ar["dp"] = cr.dispersion_dp(Xg[:, sel], Lg)["dp"].values
    ar["grupo"] = nombre
    return ar


y_et = int(
    NG[NG.fuente == "eltiempo"]
    .sort_values("tokens")
    .iloc[len(NG[NG.fuente == "eltiempo"]) // 2]["año"]
)
sem = NG[(NG.fuente == "semana") & (NG.año < 1990)].sort_values("tokens")
y_se = (
    int(sem.iloc[0]["año"])
    if len(sem)
    else int(NG[NG.fuente == "semana"].sort_values("tokens").iloc[0]["año"])
)
with cr.cronometro(TIEMPOS, "ic_demo"):
    AR = pd.concat(
        [
            d
            for d in [
                demo_ic("eltiempo_base", y_et, f"eltiempo {y_et}"),
                demo_ic("semana_base", y_se, f"semana {y_se}"),
            ]
            if d is not None
        ]
    )
for g, s in AR.groupby("grupo"):
    nmin, tb = cr.conteo_minimo(s, umbral=cfg.hw_rel_max)
    print(
        f"\n{g}: conteo mínimo (semiancho relativo bootstrap ≤ {cfg.hw_rel_max}) = {nmin}"
    )
    print(tb.round(3).to_string())
CONTEO_MIN = cr.conteo_minimo(
    AR[AR.grupo.str.startswith("eltiempo")], umbral=cfg.hw_rel_max
)[0]
print(
    "\nSpearman(cociente bootstrap/Poisson, DP), todos los conteos:",
    round(stats.spearmanr(AR["cociente"], AR["dp"], nan_policy="omit").statistic, 3),
)
# Con n < 10 la comparación está confundida: con 1–3 apariciones la DP vale ≈ 1 por construcción,
# Garwood es conservador y el bootstrap percentil se degenera (pocos docs distintos que remuestrear).
AR10 = AR[AR["n"] >= 10]
BINS10 = [10, 20, 50, 100, 200, 500, 10**9]
print("\nCociente bootstrap/Poisson (mediana) por bin de conteo, n ≥ 10:")
print(
    AR10.groupby(["grupo", pd.cut(AR10["n"], BINS10, right=False)], observed=True)[
        "cociente"
    ]
    .median()
    .unstack(0)
    .round(2)
    .to_string()
)
print(
    "Spearman(cociente, DP) dentro de cada bin de conteo (n ≥ 10):",
    {
        str(b): round(stats.spearmanr(g["cociente"], g["dp"]).statistic, 3)
        for b, g in AR10.groupby(
            pd.cut(AR10["n"], [10, 50, 200, 10**9], right=False), observed=True
        )
        if len(g) > 10
    },
)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for g, s in AR.groupby("grupo"):
    ax[0].scatter(s["n"], s["hw_rel_boot"], s=4, alpha=0.4, label=f"{g} bootstrap")
ss = AR.sort_values("n")
ax[0].plot(ss["n"], ss["hw_rel_poisson"], c="k", lw=1, label="Poisson")
ax[0].axhline(cfg.hw_rel_max, ls="--", c="r", lw=0.8)
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_xlabel("apariciones en el año")
ax[0].set_ylabel("semiancho relativo del IC 95 %")
ax[0].legend(fontsize=7)
ax[1].scatter(AR["dp"], AR["cociente"], s=4, alpha=0.4)
ax[1].set_xlabel("DP (concentración)")
ax[1].set_ylabel("ancho bootstrap / ancho Poisson")
guardar(fig, "fig_ic_poisson_bootstrap")

### 2.3 Concentración: pmw frente a frecuencia documental y dispersión

El pmw cuenta apariciones; una palabra repetida en pocos documentos (un caso judicial, un deportista, la sigla de un trámite) puede tener el mismo pmw que una palabra repartida en miles de notas. Se compara con la **frecuencia documental por mil documentos** y con la **DP de Gries (2008)** (0 = repartida como el corpus, → 1 = concentrada), con los documentos como partes. La D de Juilland exige partes de igual tamaño, por eso se usa DP.

In [ ]:
with cr.cronometro(TIEMPOS, "dispersion"):
    m = SERIES["eltiempo_base"]
    DISP = cr.dispersion_dp(Xf[m][:, ok_terms], L_forma[m])
DISP.insert(0, "termino", vf[ok_terms])
DISP["pmw"] = DISP["n"] / L_forma[m].sum() * 1e6
DISP["zipf"] = cr.zipf(DISP["pmw"])
DISP["df_por_mil"] = DISP["df"] / m.sum() * 1000
DISP["contenido"] = DISP["termino"].map(es_contenido)
DD = DISP[
    DISP.contenido & (DISP.n >= max(cfg.min_total_export, 50 if not FAST else 5))
].copy()
DD["r_pmw"] = DD["pmw"].rank(ascending=False)
DD["r_df"] = DD["df_por_mil"].rank(ascending=False)
DD["log_brecha"] = np.log2(
    DD["r_df"] / DD["r_pmw"]
)  # > 0: el pmw la pone más arriba de lo que su difusión justifica
print(
    "Spearman(pmw, df por mil) =",
    round(stats.spearmanr(DD.pmw, DD.df_por_mil).statistic, 3),
    "| Spearman(pmw, DP) =",
    round(stats.spearmanr(DD.pmw, DD.dp).statistic, 3),
)
cols = ["termino", "n", "pmw", "zipf", "df_por_mil", "dp", "r_pmw", "r_df"]
print("\nInfladas por concentración (pmw alto, pocos documentos):")
print(
    DD[DD.zipf >= 3]
    .sort_values("log_brecha", ascending=False)
    .head(20)[cols]
    .round(3)
    .to_string(index=False)
)
print("\nRepartidas (difusión mayor que lo que sugiere el pmw):")
print(
    DD[DD.zipf >= 3]
    .sort_values("log_brecha")
    .head(10)[cols]
    .round(3)
    .to_string(index=False)
)
# Con 50.000 documentos como partes la DP se satura (casi todo > 0,85): a igual Zipf se compara
# mejor la frecuencia documental. Pares con el mismo Zipf (redondeado a 0,1) y razón de df ≥ 5×.
print("\nDP por banda de Zipf (cuantiles 10/50/90):")
print(
    DD.groupby(cr.banda(DD["zipf"], cfg.bandas))["dp"]
    .quantile([0.1, 0.5, 0.9])
    .unstack()
    .round(3)
    .to_string()
)
DD["zr"] = DD["zipf"].round(1)
pares = []
for z, g in DD[DD.zipf.between(3, 4.5)].groupby("zr"):
    if len(g) >= 2:
        a, b = g.loc[g.df_por_mil.idxmax()], g.loc[g.df_por_mil.idxmin()]
        if a.df_por_mil >= 5 * b.df_por_mil:
            pares.append(
                {
                    "zipf": z,
                    "repartida": a.termino,
                    "df_rep": round(a.df_por_mil, 2),
                    "dp_rep": round(a.dp, 3),
                    "concentrada": b.termino,
                    "df_con": round(b.df_por_mil, 2),
                    "dp_con": round(b.dp, 3),
                    "razon_df": round(a.df_por_mil / b.df_por_mil, 1),
                }
            )
print(
    "\nMismo Zipf, frecuencia documental muy distinta:\n",
    pd.DataFrame(pares).to_string(index=False),
)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.scatter(DD["zipf"], DD["dp"], s=2, alpha=0.3)
ax.set_xlabel("Zipf (El Tiempo base)")
ax.set_ylabel("DP de Gries")
ax.set_title("A igual Zipf, la concentración varía mucho")
guardar(fig, "fig_zipf_vs_dp")

### 2.4 Dependencia del denominador

El mismo conteo da pmw distintos según el denominador: todos los tokens (con stopwords) o solo los lemas de contenido. El Zipf se desplaza en log₁₀(N_tokens / N_contenido), y ese desplazamiento **cambia por año y fuente** porque la proporción de palabras de contenido depende del formato (breves, titulares). Una tendencia en pmw puede ser, en parte, una tendencia del denominador.

In [ ]:
DEN = pd.DataFrame(
    {
        "fuente": [CORTO[f] for f in fuente],
        "año": año,
        "tok": L_forma,
        "cont": L_cont,
        "base": base,
        "breve": dfp["es_breve"].values,
    }
)
den = (
    DEN.groupby(["fuente", "año"])
    .apply(
        lambda g: pd.Series(
            {
                "frac_contenido_base": g.loc[g.base, "cont"].sum()
                / max(g.loc[g.base, "tok"].sum(), 1),
                "frac_contenido_todos": g["cont"].sum() / max(g["tok"].sum(), 1),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
den["desplazamiento_zipf_base"] = np.log10(1 / den["frac_contenido_base"])
den["desplazamiento_zipf_todos"] = np.log10(1 / den["frac_contenido_todos"])
print(
    den.groupby("fuente")[["desplazamiento_zipf_base", "desplazamiento_zipf_todos"]]
    .agg(["min", "max"])
    .round(3)
)
# Cuántas formas de contenido cambian de banda al pasar al denominador de contenido (El Tiempo, base)
pe = PZ[(PZ.fuente == "eltiempo") & PZ.termino.map(es_contenido)].merge(
    den[den.fuente == "eltiempo"][["año", "desplazamiento_zipf_base"]], on="año"
)
pe["zipf_cont"] = pe["zipf"] + pe["desplazamiento_zipf_base"]
pe["banda_cont"] = cr.banda(pe["zipf_cont"], cfg.bandas)
print(
    "\nFormas de contenido que cambian de banda con el denominador de contenido, por año (El Tiempo):"
)
print(
    pe.groupby("año")
    .apply(lambda g: (g.banda != g.banda_cont).mean(), include_groups=False)
    .round(3)
    .to_string()
)
fig, ax = plt.subplots(figsize=(7, 3.5))
for f, g in den.groupby("fuente"):
    ax.plot(
        g["año"], g["desplazamiento_zipf_base"], marker="o", ms=3, label=f"{f} base"
    )
    ax.plot(g["año"], g["desplazamiento_zipf_todos"], ls=":", label=f"{f} todos")
ax.axvline(cfg.año_formato, c="grey", lw=0.8)
ax.set_ylabel("Δ Zipf (contenido − todos)")
ax.legend(fontsize=7)
guardar(fig, "fig_denominador")

### 2.5 Efecto de los nombres propios

Una forma se marca como **propia** si ≥ 50 % de sus apariciones caen dentro de entidades (PER/ORG/LOC) del NER de S0. Los nombres propios ocupan el extremo frecuente del vocabulario de contenido y cambian con la coyuntura (presidentes, guerrillas, equipos): cualquier ranking de "palabras de la época" sin separarlos mide personajes, no lenguaje.

In [ ]:
FR_ENT = cr.frac_en_entidades(dfp["entidades"], dict(zip(vf, tot_f)))
DISP["frac_entidad"] = DISP["termino"].map(FR_ENT).fillna(0)
DISP["propio"] = DISP["frac_entidad"] >= cfg.umbral_propio
DISP.to_csv(RES / "dispersion.csv", index=False)
dc = DISP[DISP.contenido]
for k in [50, 200, 1000]:
    top = dc.nlargest(k, "pmw")
    print(
        f"top-{k} formas de contenido por pmw (El Tiempo base): {top.propio.mean():.0%} propias"
    )
dc = dc.assign(banda=cr.banda(dc["zipf"], cfg.bandas))
print("\nPropias por banda (tipos y tokens):")
print(
    dc.groupby("banda")
    .apply(
        lambda g: pd.Series(
            {"tipos": g.propio.mean(), "tokens": g.loc[g.propio, "n"].sum() / g.n.sum()}
        ),
        include_groups=False,
    )
    .round(3)
)
print("\nTop-20 con propios:", " ".join(dc.nlargest(20, "pmw").termino))
print("Top-20 sin propios:", " ".join(dc[~dc.propio].nlargest(20, "pmw").termino))
prop_set = set(DISP.loc[DISP.propio, "termino"])
pp = PZ[PZ.termino.map(es_contenido)].assign(propio=lambda x: x.termino.isin(prop_set))
serie_prop = (
    pp.groupby(["fuente", "año"])
    .apply(lambda g: g.loc[g.propio, "n"].sum() / g.n.sum(), include_groups=False)
    .unstack(0)
)
print(
    "\nFracción de tokens de contenido que son nombres propios:\n",
    serie_prop.round(3).to_string(),
)

### 2.6 Rareza relativa al español general (wordfreq)

`wordfreq` (Speer 2022) da el Zipf de cada palabra en una mezcla de Wikipedia, subtítulos, noticias, libros y redes. **Implicaciones**: es una referencia **posterior** a 2011 y de varios géneros, así que el vocabulario reciente parece "menos raro" por anacronismo y el registro de prensa parece "raro" frente a subtítulos y redes. Δ = Zipf(prensa) − Zipf(referencia); Zipf de referencia 0 = ausente.

In [ ]:
with cr.cronometro(TIEMPOS, "wordfreq"):
    terms_wf = sorted(
        set(DISP.loc[DISP.contenido & (DISP.n >= cfg.min_total_export), "termino"])
        | set(
            PZ.loc[
                (PZ.fuente == "semana")
                & (PZ.n >= cfg.min_total_export)
                & PZ.termino.map(es_contenido),
                "termino",
            ]
        )
    )
    ZREF = cr.zipf_referencia(terms_wf)
w = DISP[DISP.termino.isin(ZREF.index)].assign(zipf_ref=lambda x: x.termino.map(ZREF))
w["delta"] = w["zipf"] - w["zipf_ref"]
print(
    "Ausentes de wordfreq (no propias, por pmw):",
    " ".join(w[(w.zipf_ref == 0) & ~w.propio].nlargest(25, "pmw").termino),
)
print("\nPropias de la prensa (Δ alto, presentes en la referencia, no propias):")
print(
    w[(w.zipf_ref > 0) & ~w.propio]
    .nlargest(25, "delta")[["termino", "zipf", "zipf_ref", "delta", "dp"]]
    .round(2)
    .to_string(index=False)
)
print("\nMás frecuentes en el español general que en la prensa:")
print(
    w[(w.zipf_ref > 0) & ~w.propio]
    .nsmallest(15, "delta")[["termino", "zipf", "zipf_ref", "delta"]]
    .round(2)
    .to_string(index=False)
)
pr = PZ[PZ.termino.isin(ZREF.index)].assign(
    zipf_ref=lambda x: x.termino.map(ZREF), propio=lambda x: x.termino.isin(prop_set)
)


def resumen_ref(g):
    g2 = g[~g.propio]
    if g2.n.sum() == 0:
        return pd.Series(
            {
                "zipf_ref_medio": np.nan,
                "frac_tok_raros_ref": np.nan,
                "frac_tok_ausentes_ref": np.nan,
            }
        )
    return pd.Series(
        {
            "zipf_ref_medio": np.average(g2.zipf_ref, weights=g2.n),
            "frac_tok_raros_ref": g2.loc[g2.zipf_ref < cfg.bandas[0], "n"].sum()
            / g2.n.sum(),
            "frac_tok_ausentes_ref": g2.loc[g2.zipf_ref == 0, "n"].sum() / g2.n.sum(),
        }
    )


REF_AÑO = (
    pr.groupby(["fuente", "año"]).apply(resumen_ref, include_groups=False).reset_index()
)
print(
    "\nPendiente por año (El Tiempo), zipf_ref_medio:",
    round(
        np.polyfit(
            *REF_AÑO[REF_AÑO.fuente == "eltiempo"][["año", "zipf_ref_medio"]].values.T,
            1,
        )[0],
        4,
    ),
)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for f, g in REF_AÑO.groupby("fuente"):
    ax[0].plot(g.año, g.zipf_ref_medio, marker="o", ms=3, label=f)
    ax[1].plot(g.año, g.frac_tok_raros_ref, marker="o", ms=3, label=f)
ax[0].set_title("Zipf medio en el español general (tokens no propios)")
ax[1].set_title("Fracción de tokens raros en el español general")
for a in ax:
    a.axvline(cfg.año_formato, c="grey", lw=0.8)
    a.legend()
guardar(fig, "fig_wordfreq")

## 3. Variantes que refieren a lo mismo (observación a)

Para cada conjunto congelado y cada serie: **cuota** de cada variante sobre el total del conjunto (razón de sumas) con **IC 95 % por bootstrap por documento** (B = 1000), y NaN explícito si el conjunto tiene menos de `min_ocurr_año` apariciones en el año. Controles: El Tiempo con breves, Semana, Dinero y **El Tiempo estandarizado por tópico dominante** (misma mezcla de tópicos todos los años).

In [ ]:
# ⏱ ~1–2 min: búsqueda de variantes en tokens_forma
with cr.cronometro(TIEMPOS, "contar_variantes"):
    TXT = cr.textos_unidos(dfp["tokens_forma"])
    CONT = {c: cr.contar_patrones(TXT, d["variantes"]) for c, d in VARIANTES.items()}
    CONT_ESTE = cr.contar_patrones(TXT, {"éste": ESTE_TILDE})
tot = pd.DataFrame(
    [
        {
            "conjunto": c,
            "variante": v,
            "apariciones": int(CONT[c][v].sum()) if len(CONT[c]) else 0,
            "docs": int((CONT[c][v] > 0).sum()) if len(CONT[c]) else 0,
        }
        for c, d in VARIANTES.items()
        for v in d["variantes"]
    ]
)
print(tot.to_string(index=False))

In [ ]:
# ⏱ ~2–3 min: cuotas con bootstrap por serie
SERIES_VAR = ["eltiempo_base", "eltiempo_todos", "semana_base", "dinero_base"]
partes, REPS = [], {}
with cr.cronometro(TIEMPOS, "cuotas"):
    for c in VARIANTES:
        for s in SERIES_VAR:
            etq = np.where(SERIES[s], año, None)
            t, r = cr.cuotas(CONT[c], etq, cfg.b_boot, cfg.seed, cfg.min_ocurr_año)
            if len(t):
                partes.append(
                    t.rename(columns={"grupo": "año"}).assign(conjunto=c, serie=s)
                )
                REPS[(c, s)] = r
        etq = np.where(SERIES["eltiempo_base"], año, None)
        t = cr.cuota_estandarizada(
            CONT[c], etq, dfp["topico"].values, cfg.min_ocurr_año
        )
        if len(t):
            partes.append(
                t.rename(columns={"grupo": "año"}).assign(
                    conjunto=c, serie="eltiempo_base_std_topico"
                )
            )
CUO = pd.concat(partes, ignore_index=True)
CUO["año"] = CUO["año"].astype(int)
CUO["categoria"] = CUO.conjunto.map(lambda c: VARIANTES[c]["categoria"])
CUO["tipo"] = CUO.conjunto.map(lambda c: VARIANTES[c]["tipo"])
CUO.to_csv(RES / "variantes_cuotas.csv", index=False)
print("Años con cuota válida (por conjunto y serie):")
print(
    CUO.dropna(subset=["cuota"])
    .groupby(["conjunto", "serie"])["año"]
    .nunique()
    .unstack()
    .fillna(0)
    .astype(int)
    .to_string()
)

**Año de cruce** (regla declarada): primer año desde el cual la variante nueva supera a la vieja durante `cruce_años` = 3 años válidos consecutivos, habiendo estado debajo antes. Si la nueva domina desde el primer año válido, no hay cruce que fechar. El IC sale de recalcular el cruce en cada réplica bootstrap (`frac_reps_con_cruce` dice en qué fracción existe). **Detector de cambio**: `agenda.binseg` sobre √cuota, como en el 08. Toda fecha a ±1 de 2006 se marca (`cerca_formato`).

In [ ]:
filas = []
for c, d in VARIANTES.items():
    for s in SERIES_VAR + ["eltiempo_base_std_topico"]:
        t = CUO[(CUO.conjunto == c) & (CUO.serie == s)]
        base_f = {
            "conjunto": c,
            "categoria": d["categoria"],
            "tipo": d["tipo"],
            "serie": s,
        }
        if t["cuota"].notna().sum() == 0:
            filas.append(
                {**base_f, "estado": "datos_insuficientes", "cambios": [], "fuerza": []}
            )
            continue
        años_s = sorted(t.loc[t.cuota.notna(), "año"].unique())
        cb = cr.cambios_cuota(t, cfg)
        cx = (
            cr.cruces(
                t,
                REPS.get((c, s), {}),
                d["cruce"],
                list(range(min(años_s), max(años_s) + 1)),
                list(d["variantes"]),
                cfg.cruce_años,
                año_formato=cfg.año_formato,
            )
            if d["cruce"]
            else {"estado": "sin_hipotesis_de_cruce"}
        )
        filas.append(
            {
                **base_f,
                **cx,
                "años_validos": f"{min(años_s)}–{max(años_s)} ({len(años_s)})",
                "cambios": cb["cambios"],
                "fuerza": cb["fuerza"],
                "cambio_cerca_formato": any(
                    abs(y - cfg.año_formato) <= 1 for y in cb["cambios"]
                ),
            }
        )
CRU = pd.DataFrame(filas)
CRU.to_csv(RES / "cruces_variantes.csv", index=False)
# Concordancia de los controles con la serie principal (±1 año)
piv = CRU.pivot(index="conjunto", columns="serie", values="cruce")
LECT = pd.DataFrame(
    {
        "tipo": [VARIANTES[c]["tipo"] for c in piv.index],
        "cruce_ET_base": piv.get("eltiempo_base"),
        "con_breves": piv.get("eltiempo_todos"),
        "std_topico": piv.get("eltiempo_base_std_topico"),
        "semana": piv.get("semana_base"),
    },
    index=piv.index,
)
cerca = lambda a, b: np.isfinite(a) and np.isfinite(b) and abs(a - b) <= 1
LECT["robusto_breves"] = [
    cerca(a, b) for a, b in zip(LECT.cruce_ET_base, LECT.con_breves)
]
LECT["robusto_topico"] = [
    cerca(a, b) for a, b in zip(LECT.cruce_ET_base, LECT.std_topico)
]
LECT["semana_concuerda"] = [
    cerca(a, b) for a, b in zip(LECT.cruce_ET_base, LECT.semana)
]
LECT["cerca_2006"] = [
    np.isfinite(a) and abs(a - cfg.año_formato) <= 1 for a in LECT.cruce_ET_base
]
print(
    CRU[CRU.serie == "eltiempo_base"][
        [
            "conjunto",
            "tipo",
            "estado",
            "cruce",
            "lo",
            "hi",
            "frac_reps_con_cruce",
            "cerca_formato",
            "cambios",
            "cambio_cerca_formato",
        ]
    ].to_string(index=False)
)
print("\nCruces y controles:\n", LECT.to_string())
print(
    "\nCambios detectados por serie:\n",
    CRU.pivot(index="conjunto", columns="serie", values="cambios").to_string(),
)

In [ ]:
cs = list(VARIANTES)
nc = 4
nr = int(np.ceil(len(cs) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(17, 3.0 * nr), sharex=True)
for ax, c in zip(axs.ravel(), cs):
    for k, v in enumerate(VARIANTES[c]["variantes"]):
        col = f"C{k}"
        t = CUO[
            (CUO.conjunto == c) & (CUO.serie == "eltiempo_base") & (CUO.variante == v)
        ].sort_values("año")
        ax.plot(t.año, t.cuota, c=col, lw=1.4, label=v)
        ax.fill_between(t.año, t.lo, t.hi, color=col, alpha=0.2)
        s = CUO[
            (CUO.conjunto == c) & (CUO.serie == "semana_base") & (CUO.variante == v)
        ].sort_values("año")
        ax.plot(s.año, s.cuota, c=col, ls=":", lw=1)
        e = CUO[
            (CUO.conjunto == c)
            & (CUO.serie == "eltiempo_base_std_topico")
            & (CUO.variante == v)
        ].sort_values("año")
        ax.plot(e.año, e.cuota, c=col, ls="--", lw=0.7)
    ax.axvline(cfg.año_formato, c="grey", lw=0.8)
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"{c} [{VARIANTES[c]['tipo']}]", fontsize=9)
    ax.legend(fontsize=6, loc="best")
for ax in axs.ravel()[len(cs) :]:
    ax.axis("off")
fig.suptitle(
    "Cuota por variante · línea: El Tiempo base con IC 95 % · punteada: Semana · discontinua: El Tiempo estandarizado por tópico · gris: 2006",
    y=1.0,
)
guardar(fig, "fig_variantes_cuotas")

# éste/ésta pmw (no es cuota) y sólo como pmw, para complementar la cota inferior
filas = []
for s in ["eltiempo_base", "semana_base"]:
    for y in sorted(set(año[SERIES[s]])):
        idx = np.where(SERIES[s] & (año == y))[0]
        if L_forma[idx].sum() < cfg.min_tokens_grupo:
            continue
        Xg = np.column_stack(
            [
                CONT_ESTE.reindex(idx, fill_value=0)["éste"].values,
                CONT["solo_tilde"].reindex(idx, fill_value=0)["sólo"].values,
            ]
        )
        lo, hi, _ = cr.bootstrap_tasa(Xg, L_forma[idx], cfg.b_boot, cfg.seed)
        pm = Xg.sum(0) / L_forma[idx].sum() * 1e6
        for k, t in enumerate(["éste/ésta (tilde)", "sólo (tilde)"]):
            filas.append(
                {
                    "serie": s,
                    "año": y,
                    "termino": t,
                    "pmw": pm[k],
                    "lo": lo[k],
                    "hi": hi[k],
                }
            )
TILDES = pd.DataFrame(filas)
TILDES.to_csv(RES / "tildes_pmw.csv", index=False)
print(
    TILDES.pivot_table(index="año", columns=["serie", "termino"], values="pmw")
    .round(1)
    .to_string()
)

**Contextos** (3 por variante, período y fuente, al azar con semilla fija): sirven para verificar que las variantes de un conjunto `reemplazo` refieren a lo mismo y para mostrar la valoración distinta en los conjuntos `marco`. Incluyen breves (son ejemplos, no medidas). Semana 1982–1989 aparece como período `S80`.

In [ ]:
per_ctx = np.where((fuente == SEMANA) & (año < cfg.et_desde), "S80", per)
filas = []
for c, d in VARIANTES.items():
    rx, mapa = cr.compilar(d["variantes"])
    C = CONT[c]
    for v in d["variantes"]:
        docs_v = C.index[C[v] > 0].values if len(C) else np.array([], int)
        for f in [ET, SEMANA]:
            for p in ["S80"] + [x[0] for x in cfg.periodos]:
                pos = [i for i in docs_v if fuente[i] == f and per_ctx[i] == p]
                if not pos:
                    continue
                ej = cr.contextos(
                    TXT,
                    ids,
                    pos,
                    rx,
                    mapa,
                    v,
                    cfg.n_contextos,
                    cfg.ventana_kwic,
                    cfg.seed,
                )
                for e in ej:
                    filas.append(
                        {
                            "conjunto": c,
                            "tipo": d["tipo"],
                            "variante": v,
                            "fuente": CORTO[f],
                            "periodo": p,
                            "contexto": e,
                        }
                    )
CTX = pd.DataFrame(filas)
CTX.to_csv(RES / "contextos_variantes.csv", index=False)
for c in ["gentilicio_eeuu", "insurgencia", "secuestro", "computador"]:
    print(f"\n=== {c} ===")
    print(
        CTX[(CTX.conjunto == c) & (CTX.fuente == "eltiempo")]
        .groupby(["variante", "periodo"])
        .head(1)[["variante", "periodo", "contexto"]]
        .to_string(index=False)
    )

**Lectura** (se completa con las tablas de arriba, sin cambiar las reglas):
* En los conjuntos `reemplazo`, un cruce cuenta como reemplazo léxico si es **robusto a breves y a tópico**. Si además cae a ±1 de 2006 y Semana no concuerda, queda **confundido con el cambio de archivo**.
* En los conjuntos `marco` no hay cruce: se describen las cuotas y los cambios detectados como **cambios de denominación con valoración distinta**, verificados con los contextos.

## 4. Cambio semántico: word2vec por período (El Tiempo)

* **Períodos** (alineados con los regímenes del 08): P1 1990–1994 | P2 1995–1999 (1995 = cambio robusto) | P3 2000–2005 (formato de archivo con lead duplicado) | P4 2006–2011. **El corte de 2006 coincide con el cambio de formato de El Tiempo**: el desplazamiento P3→P4 se lee con esa reserva y se contrasta con la corrida con breves.
* **Tokens**: `lemas_contenido` (N, A, V, P), partidos en `cortes_contenido`, con los bigramas de la lista blanca, `estados_unidos` y el mapa de plurales del 08.
* **Mismo tamaño**: cada período se submuestrea por documentos hasta los tokens del período más chico.
* **Ruido**: 5 semillas por período. El desplazamiento de una palabra **cuenta** solo si el percentil 5 de las distancias entre períodos supera el percentil 95 de las distancias entre semillas del mismo período.
* **Dos corridas**: `base` (sin breves ni cartas) y `con_breves` (todo).

In [ ]:
LB = frozenset(
    agenda.leer_lista_blanca(utils.data_dir() / "lista_blanca_con_propn.txt")
)
mp = pd.read_csv(R08 / "mapa_plurales.csv")
MAPA = dict(zip(mp["plural"], mp["singular"]))
RENOM = {"unidos": "estados_unidos"}
with cr.cronometro(TIEMPOS, "secuencias"):
    SEGS = [
        cr.secuencias_lema(l, p, c, LB, RENOM, MAPA)
        for l, p, c in zip(
            dfp["lemas_contenido"], dfp["pos_contenido"], dfp["cortes_contenido"]
        )
    ]
    POSDOM = cr.pos_dominante(dfp["lemas_contenido"], dfp["pos_contenido"])
LB_TEMATICOS = {"acuerdo_paz", "medio_ambiente", "medio_comunicación"}
PROPIOS_LEMA = (
    {l for l, p in POSDOM.items() if p == "P"}
    | (set(LB) - LB_TEMATICOS)
    | {"estados_unidos"}
)
VAR_W2V = {"base": SERIES["eltiempo_base"], "con_breves": SERIES["eltiempo_todos"]}
CORP, CONTEO, OBJ = {}, {}, {}
for v, m in VAR_W2V.items():
    CORP[v], CONTEO[v], OBJ[v] = cr.corpus_periodos(SEGS, per, m, cfg.seed)
    print(
        v,
        "| tokens por período (igualados):",
        OBJ[v],
        "| tramos:",
        {p: len(x) for p, x in CORP[v].items()},
        "| vocabulario:",
        {p: len(c) for p, c in CONTEO[v].items()},
    )


def _rel(c):
    tot = sum(c.values())
    return {w: n / tot * 1e6 for w, n in c.items()}


FREC = {v: {p: _rel(c) for p, c in CONTEO[v].items()} for v in CONTEO}

In [ ]:
# ⏱ ~10–15 min por variante la primera vez (luego se carga del caché)
MODELOS = {v: {} for v in CORP}
with cr.cronometro(TIEMPOS, "word2vec"):
    for v in CORP:
        for p in sorted(CORP[v]):
            MODELOS[v][p] = [
                cr.entrenar_w2v(
                    CORP[v][p], s, cfg, CACHE / f"{HASH}_w2v_{v}_{p}_s{s}.kv"
                )
                for s in cfg.w2v_semillas
            ]
            print(v, p, "vocab:", len(MODELOS[v][p][0].key_to_index))

**Umbral de vocabulario por evidencia.** Las palabras raras tienen vectores ruidosos: su distancia entre semillas del mismo período (`d_dentro`) es alta. **Regla declarada**: el umbral más bajo cuyo bin de frecuencia [uᵢ, uᵢ₊₁) tiene mediana de `d_dentro` < 0,5 × mediana global de `d_entre`.

In [ ]:
umin = min(cfg.umbrales_vocab)
VOC0 = cr.vocab_comun(MODELOS["base"], CONTEO["base"], umin)
D0 = cr.desplazamiento(MODELOS["base"], VOC0, cfg.q_sep)
fmin = pd.Series({w: min(CONTEO["base"][p][w] for p in CONTEO["base"]) for w in VOC0})
U, TAB_U = cr.elegir_umbral_vocab(D0, fmin, cfg.umbrales_vocab, cfg.frac_ruido_vocab)
print(TAB_U.round(4).to_string(index=False))
if not np.isfinite(U):
    U = max(cfg.umbrales_vocab)
    print("Ningún umbral cumple la regla: se usa el más alto, y se declara.")
VOC = cr.vocab_comun(MODELOS["base"], CONTEO["base"], int(U))
print(
    f"Umbral elegido: {U} apariciones en cada período → vocabulario común de {len(VOC):,} palabras"
)

In [ ]:
with cr.cronometro(TIEMPOS, "desplazamiento"):
    DESP = cr.desplazamiento(MODELOS["base"], VOC, cfg.q_sep)
    DESP, CF = cr.control_frecuencia(DESP, FREC["base"])
DESP["propio"] = DESP["palabra"].isin(PROPIOS_LEMA)
print("Control de frecuencia por transición:")
print(pd.DataFrame(CF).T.to_string())
print(
    "\nPalabras que cuentan (superan el ruido entre semillas):",
    DESP.groupby("transicion")["cuenta"].agg(["sum", "mean"]).round(3).to_dict(),
)
cols = ["palabra", "d_entre", "d_dentro", "efecto", "efecto_resid", "log_f", "dlog_f"]
for tr, s in DESP.groupby("transicion"):
    s = s[s.cuenta & ~s.propio]
    print(f"\n{tr} · top-15 por efecto:", " ".join(s.nlargest(15, "efecto").palabra))
    print(
        f"{tr} · top-15 por efecto residual (sin frecuencia):",
        " ".join(s.nlargest(15, "efecto_resid").palabra),
    )
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (tr, s) in zip(axs, DESP.groupby("transicion")):
    ax.scatter(s.log_f, s.efecto, s=2, alpha=0.3, c=np.where(s.cuenta, "C3", "C0"))
    ax.set_title(f"{tr} (rojo: cuenta)")
    ax.set_xlabel("log10 pmw medio")
    ax.set_ylabel("d_entre − d_dentro")
guardar(fig, "fig_desplazamiento_vs_frecuencia")

**Medida alternativa sin alineación** (vecindario local, Hamilton et al. 2016b, k = 25): no depende de Procrustes. Si los dos rankings concuerdan, el desplazamiento no es un artefacto de la alineación.

In [ ]:
# ⏱ ~2 min
with cr.cronometro(TIEMPOS, "vecindario_local"):
    LOC = cr.desplazamiento_local(MODELOS["base"], VOC, cfg.k_vecinos)
DESP = DESP.merge(LOC, on=["palabra", "transicion"], how="left")
for tr, s in DESP[~DESP.propio].groupby("transicion"):
    t50g = set(s.nlargest(50, "efecto").palabra)
    t50l = set(s.nlargest(50, "efecto_local").palabra)
    print(
        f"{tr}: Spearman(global, local) = {stats.spearmanr(s.efecto, s.efecto_local).statistic:.3f} | "
        f"solape top-50 = {len(t50g & t50l)} | local con ruido > 0: {(s.efecto_local > 0).mean():.1%}"
    )

In [ ]:
# Sensibilidad: misma medida con breves
VOCb = cr.vocab_comun(MODELOS["con_breves"], CONTEO["con_breves"], int(U))
DESPb = cr.desplazamiento(MODELOS["con_breves"], VOCb, cfg.q_sep)
cmp_b = DESP.merge(
    DESPb[["palabra", "transicion", "efecto", "cuenta"]],
    on=["palabra", "transicion"],
    suffixes=("", "_cb"),
)
for tr, s in cmp_b[~cmp_b.propio].groupby("transicion"):
    print(
        f"{tr}: Spearman(base, con breves) = {stats.spearmanr(s.efecto, s.efecto_cb).statistic:.3f} | "
        f"solape top-50 = {len(set(s.nlargest(50, 'efecto').palabra) & set(s.nlargest(50, 'efecto_cb').palabra))} | "
        f"cuentan en ambas / en base: {(s.cuenta & s.cuenta_cb).sum()}/{s.cuenta.sum()}"
    )
out = pd.concat(
    [
        DESP.assign(variante="base"),
        DESPb.assign(
            variante="con_breves", propio=lambda x: x.palabra.isin(PROPIOS_LEMA)
        ),
    ]
)
out.to_csv(RES / "desplazamiento_semantico.csv", index=False)

**Vecinos por período** para las semillas declaradas y los 10 más desplazados (que cuentan y no son propios) de cada transición. Solo vecinos estables: en el top-10 de ≥ 3 de las 5 semillas. Si una palabra no está en el vocabulario de un período, la lista queda **vacía y marcada**, nunca rellenada.

In [ ]:
desplazados = []
for tr, s in DESP[DESP.cuenta & ~DESP.propio].groupby("transicion"):
    desplazados += s.nlargest(10, "efecto").palabra.tolist()
foco = [(w, "semilla") for w in SEMILLAS_SEM] + [
    (w, "desplazado") for w in dict.fromkeys(desplazados) if w not in SEMILLAS_SEM
]
filas, por_periodo = [], defaultdict(dict)
for w, origen in foco:
    for p in sorted(MODELOS["base"]):
        vec = cr.vecinos_estables(
            MODELOS["base"][p], w, cfg.k_mostrar, cfg.min_semillas_vecino
        )
        por_periodo[p][w] = vec
        if not vec:
            filas.append(
                {
                    "palabra": w,
                    "origen": origen,
                    "periodo": p,
                    "rango": None,
                    "vecino": None,
                    "n_semillas": 0,
                    "sim": np.nan,
                    "nota": "fuera de vocabulario o sin vecinos estables",
                }
            )
        for r, (v, n, sim) in enumerate(vec, 1):
            filas.append(
                {
                    "palabra": w,
                    "origen": origen,
                    "periodo": p,
                    "rango": r,
                    "vecino": v,
                    "n_semillas": n,
                    "sim": sim,
                    "nota": "",
                }
            )
for p, listas in por_periodo.items():
    cr.verificar_no_repetidas(
        listas
    )  # bug del 05: listas idénticas para palabras distintas
VEC = pd.DataFrame(filas)
VEC.to_csv(RES / "vecinos_por_periodo.csv", index=False)
tabla = (
    VEC[VEC.rango.notna() & (VEC.rango <= 6)]
    .groupby(["palabra", "periodo"])["vecino"]
    .apply(" ".join)
    .unstack()
)
print(tabla.loc[[w for w, _ in foco if w in tabla.index]].to_string())

## 4b. Control de ruido corregido: réplicas bootstrap de documentos (post hoc, declarado)

**Por qué existe esta sección.** El control de la sección 4 (5 semillas sobre el **mismo** corpus) quedó degenerado: el 97 % del vocabulario "cuenta" como desplazado, porque la distancia entre semillas solo mide el ruido de inicialización (mediana 0,06) y no la variación por muestreo de documentos, que sí está en la distancia entre períodos (0,26). Además `d_dentro` crece con la frecuencia (ρ ≈ +0,7): las palabras raras forman una región compartida que se alinea bien entre semillas. Este diagnóstico se hizo **después** de ver resultados, así que la corrección se declara aquí como post hoc; la sección 4 se conserva tal cual.

**Cambio único:** cada una de las 5 réplicas por período se entrena sobre un **bootstrap de documentos** del período (con reemplazo, hasta el mismo número de tokens), con semilla distinta (Antoniak y Mimno 2018). La regla de decisión, el umbral de vocabulario por evidencia, la medida local y la corrida con breves son **las mismas** de la sección 4.

In [ ]:
# ⏱ ~3–4 min la primera vez (40 modelos; luego caché)
MODELOS_B = {v: {} for v in VAR_W2V}
with cr.cronometro(TIEMPOS, "word2vec_bootstrap"):
    for v, m in VAR_W2V.items():
        for k, (p, *_) in enumerate(cfg.periodos):
            idx = np.where(m & (per == p))[0]
            MODELOS_B[v][p] = [
                cr.entrenar_bootstrap(
                    SEGS,
                    idx,
                    OBJ[v],
                    r,
                    k,
                    cfg,
                    CACHE / f"{HASH}_w2vboot_{v}_{p}_r{r}.kv",
                )
                for r in range(len(cfg.w2v_semillas))
            ]
VOC0_B = cr.vocab_comun(MODELOS_B["base"], CONTEO["base"], umin)
D0_B = cr.desplazamiento(MODELOS_B["base"], VOC0_B, cfg.q_sep)
fmin_B = pd.Series(
    {w: min(CONTEO["base"][p][w] for p in CONTEO["base"]) for w in VOC0_B}
)
U_B, TAB_U_B = cr.elegir_umbral_vocab(
    D0_B, fmin_B, cfg.umbrales_vocab, cfg.frac_ruido_vocab
)
print(TAB_U_B.round(4).to_string(index=False))
if not np.isfinite(U_B):
    U_B = max(cfg.umbrales_vocab)
    print("Ningún umbral cumple la regla: se usa el más alto, y se declara.")
VOC_B = cr.vocab_comun(MODELOS_B["base"], CONTEO["base"], int(U_B))
print(f"Umbral (bootstrap): {U_B} → vocabulario común de {len(VOC_B):,} palabras")

In [ ]:
with cr.cronometro(TIEMPOS, "desplazamiento_bootstrap"):
    DESP_B = cr.desplazamiento(MODELOS_B["base"], VOC_B, cfg.q_sep)
    DESP_B, CF_B = cr.control_frecuencia(DESP_B, FREC["base"])
    DESP_B["propio"] = DESP_B["palabra"].isin(PROPIOS_LEMA)
    LOC_B = cr.desplazamiento_local(MODELOS_B["base"], VOC_B, cfg.k_vecinos)
    DESP_B = DESP_B.merge(LOC_B, on=["palabra", "transicion"], how="left")
print(
    "Semillas (sección 4) frente a bootstrap (4b): mediana d_dentro, mediana d_entre y fracción que cuenta"
)
cmp4 = pd.DataFrame(
    {
        "d_dentro_semillas": DESP.groupby("transicion")["d_dentro"].median(),
        "d_dentro_bootstrap": DESP_B.groupby("transicion")["d_dentro"].median(),
        "d_entre_semillas": DESP.groupby("transicion")["d_entre"].median(),
        "d_entre_bootstrap": DESP_B.groupby("transicion")["d_entre"].median(),
        "cuenta_semillas": DESP.groupby("transicion")["cuenta"].mean(),
        "cuenta_bootstrap": DESP_B.groupby("transicion")["cuenta"].mean(),
    }
)
print(cmp4.round(3).to_string())
print("\nControl de frecuencia (bootstrap):")
print(pd.DataFrame(CF_B).T.to_string())
for tr, s in DESP_B[~DESP_B.propio].groupby("transicion"):
    t50g, t50l = (
        set(s.nlargest(50, "efecto").palabra),
        set(s.nlargest(50, "efecto_local").palabra),
    )
    print(
        f"{tr}: Spearman(global, local) = {stats.spearmanr(s.efecto, s.efecto_local).statistic:.3f} | solape top-50 = {len(t50g & t50l)}"
    )
cols = [
    "palabra",
    "d_entre",
    "d_dentro",
    "efecto",
    "efecto_resid",
    "efecto_local",
    "log_f",
]
for tr, s in DESP_B.groupby("transicion"):
    s = s[s.cuenta & ~s.propio]
    print(
        f"\n{tr} · cuentan {len(s)} (no propias) · top-20 por efecto:\n",
        s.nlargest(20, "efecto")[cols].round(3).to_string(index=False),
    )
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (tr, s) in zip(axs, DESP_B.groupby("transicion")):
    ax.scatter(s.log_f, s.efecto, s=2, alpha=0.3, c=np.where(s.cuenta, "C3", "C0"))
    ax.set_title(f"{tr} bootstrap (rojo: cuenta)")
    ax.set_xlabel("log10 pmw medio")
    ax.set_ylabel("d_entre − d_dentro")
guardar(fig, "fig_desplazamiento_bootstrap_vs_frecuencia")

In [ ]:
# Sensibilidad con breves y vecinos estables entre réplicas bootstrap
VOCb_B = cr.vocab_comun(MODELOS_B["con_breves"], CONTEO["con_breves"], int(U_B))
DESPb_B = cr.desplazamiento(MODELOS_B["con_breves"], VOCb_B, cfg.q_sep)
cmp_bB = DESP_B.merge(
    DESPb_B[["palabra", "transicion", "efecto", "cuenta"]],
    on=["palabra", "transicion"],
    suffixes=("", "_cb"),
)
for tr, s in cmp_bB[~cmp_bB.propio].groupby("transicion"):
    print(
        f"{tr}: Spearman(base, con breves) = {stats.spearmanr(s.efecto, s.efecto_cb).statistic:.3f} | "
        f"solape top-50 = {len(set(s.nlargest(50, 'efecto').palabra) & set(s.nlargest(50, 'efecto_cb').palabra))} | "
        f"cuentan en ambas / en base: {(s.cuenta & s.cuenta_cb).sum()}/{s.cuenta.sum()}"
    )
pd.concat(
    [
        DESP_B.assign(variante="base"),
        DESPb_B.assign(
            variante="con_breves", propio=lambda x: x.palabra.isin(PROPIOS_LEMA)
        ),
    ]
).to_csv(RES / "desplazamiento_semantico_bootstrap.csv", index=False)

desp_B = []
for tr, s in DESP_B[DESP_B.cuenta & ~DESP_B.propio].groupby("transicion"):
    desp_B += s.nlargest(10, "efecto").palabra.tolist()
foco_B = [(w, "semilla") for w in SEMILLAS_SEM] + [
    (w, "desplazado") for w in dict.fromkeys(desp_B) if w not in SEMILLAS_SEM
]
filas, por_periodo = [], defaultdict(dict)
for w, origen in foco_B:
    for p in sorted(MODELOS_B["base"]):
        vec = cr.vecinos_estables(
            MODELOS_B["base"][p], w, cfg.k_mostrar, cfg.min_semillas_vecino
        )
        por_periodo[p][w] = vec
        if not vec:
            filas.append(
                {
                    "palabra": w,
                    "origen": origen,
                    "periodo": p,
                    "rango": None,
                    "vecino": None,
                    "n_semillas": 0,
                    "sim": np.nan,
                    "nota": "fuera de vocabulario o sin vecinos estables",
                }
            )
        for r, (v, n, sim) in enumerate(vec, 1):
            filas.append(
                {
                    "palabra": w,
                    "origen": origen,
                    "periodo": p,
                    "rango": r,
                    "vecino": v,
                    "n_semillas": n,
                    "sim": sim,
                    "nota": "",
                }
            )
for p, listas in por_periodo.items():
    cr.verificar_no_repetidas(listas)
VEC_B = pd.DataFrame(filas)
VEC_B.to_csv(RES / "vecinos_por_periodo_bootstrap.csv", index=False)
tabla = (
    VEC_B[VEC_B.rango.notna() & (VEC_B.rango <= 6)]
    .groupby(["palabra", "periodo"])["vecino"]
    .apply(" ".join)
    .unstack()
)
print(tabla.loc[[w for w, _ in foco_B if w in tabla.index]].to_string())

## 5. Neologismos y obsolescencia

Sobre lemas (mismo flujo que word2vec), pmw anual con todos los tokens del año como denominador (comparable con el Zipf de la sección 2). **Regla declarada** (primer uso sostenido): pmw ≥ u en el año y en ≥ 4 de los 5 siguientes, con mediana < 0,25·u en ≥ 3 años previos (si no hay 3 años previos, la palabra queda censurada: El Tiempo empieza en 1990, así que el primer neologismo fechable es 1993). La obsolescencia es la misma regla con el tiempo invertido. u = 5 pmw (≈ 7–10 apariciones por año en El Tiempo, coherente con el conteo mínimo de 2.2; con u = 1 la regla captura ruido de palabras raras). Sensibilidad: u ∈ {2; 5; 10}. Se separan entidades (POS dominante PROPN o instituciones de la lista blanca) de vocabulario común.

In [ ]:
with cr.cronometro(TIEMPOS, "neologismos"):
    TOKL = [[t for s in d for t in s] for d in SEGS]
    Xl, vl = cr.matriz_conteos(TOKL, min_df=cfg.min_df_formas)

    def pmw_años(mask):
        etq = np.where(mask, año, None)
        G, ys = cr.indicadora(etq)
        N = G @ L_forma.astype(float)
        P = pd.DataFrame((G @ Xl).toarray() / N[:, None] * 1e6, index=ys, columns=vl)
        P.loc[N < cfg.min_tokens_grupo] = np.nan
        return P

    P_ET, P_SE = pmw_años(SERIES["eltiempo_base"]), pmw_años(SERIES["semana_base"])
    cand = P_ET.columns[P_ET.max() >= min(cfg.umbrales_neo_sens)]
    filas = []
    for u in cfg.umbrales_neo_sens:
        for modo, tipo in [("primero", "neologismo"), ("ultimo", "obsolescencia")]:
            r = cr.uso_sostenido(P_ET[cand], u, cfg, modo).assign(umbral=u, tipo=tipo)
            filas.append(r)
    NEO = pd.concat(filas, ignore_index=True)
    NEO = NEO.rename(
        columns={
            "pmw_antes": "pmw_mediana_lado_ausente",
            "pmw_despues": "pmw_mediana_lado_uso",
        }
    )
    NEO["entidad"] = NEO["termino"].isin(PROPIOS_LEMA)
    # Confirmación en Semana (umbral principal)
    sem = pd.concat(
        [
            cr.uso_sostenido(
                P_SE[[c for c in cand if c in P_SE.columns]], cfg.umbral_neo, cfg, m
            ).assign(tipo=t)
            for m, t in [("primero", "neologismo"), ("ultimo", "obsolescencia")]
        ]
    )
    NEO = NEO.merge(
        sem[["termino", "tipo", "año"]].rename(columns={"año": "año_semana"}),
        on=["termino", "tipo"],
        how="left",
    )
NEO.to_csv(RES / "neologismos.csv", index=False)
print(NEO.groupby(["tipo", "umbral", "entidad"]).size().unstack().to_string())
for t in ["neologismo", "obsolescencia"]:
    a = {
        u: set(NEO[(NEO.tipo == t) & (NEO.umbral == u) & ~NEO.entidad].termino)
        for u in cfg.umbrales_neo_sens
    }
    # Jaccard queda acotado por la diferencia de tamaños; el solapamiento |A∩B|/min(|A|,|B|) no.
    print(
        t,
        "· solapamiento entre umbrales:",
        {
            f"{x}-{y}": round(len(a[x] & a[y]) / max(1, min(len(a[x]), len(a[y]))), 2)
            for x, y in zip(cfg.umbrales_neo_sens[:-1], cfg.umbrales_neo_sens[1:])
        },
        "| Jaccard:",
        {
            f"{x}-{y}": round(len(a[x] & a[y]) / max(1, len(a[x] | a[y])), 2)
            for x, y in zip(cfg.umbrales_neo_sens[:-1], cfg.umbrales_neo_sens[1:])
        },
    )
    comunes = set.intersection(*a.values())
    ya = {
        u: NEO[
            (NEO.tipo == t) & (NEO.umbral == u) & NEO.termino.isin(comunes)
        ].set_index("termino")["año"]
        for u in cfg.umbrales_neo_sens
    }
    if comunes:
        print(
            "   en los presentes con los tres umbrales, |Δ año| mediano entre u extremos:",
            float(
                (ya[cfg.umbrales_neo_sens[0]] - ya[cfg.umbrales_neo_sens[-1]])
                .abs()
                .median()
            ),
        )
prin = NEO[(NEO.umbral == cfg.umbral_neo) & ~NEO.entidad]
for t in ["neologismo", "obsolescencia"]:
    s = (
        prin[prin.tipo == t]
        .sort_values("pmw_mediana_lado_uso", ascending=False)
        .head(40)
    )
    print(f"\n{t}s comunes (top-40):")
    print(
        s[["termino", "año", "año_semana", "pmw_mediana_lado_uso"]]
        .round(1)
        .to_string(index=False)
    )
print(
    "\nPor año (comunes, umbral principal):\n",
    prin.groupby(["tipo", "año"]).size().unstack(0).fillna(0).astype(int).T.to_string(),
)

**Categorización a mano** (CSV editable, como `nombres_topicos.csv` del 08): `results/09/categorias_lexico.csv`. Completa la columna `categoria` (tecnologia, conflicto, economia, politica, cultura, otro) y vuelve a correr esta celda; las ediciones se respetan.

In [ ]:
top_cat = pd.concat(
    [
        prin[prin.tipo == t]
        .sort_values("pmw_mediana_lado_uso", ascending=False)
        .head(60)
        for t in ["neologismo", "obsolescencia"]
    ]
)
CAT, estado = cr.tabla_categorias(
    RES / "categorias_lexico.csv", top_cat[["termino", "tipo", "año", "año_semana"]]
)
print("categorias_lexico.csv:", estado)
if (CAT["categoria"] != "").any():
    print(
        CAT[CAT.categoria != ""]
        .groupby(["tipo", "categoria"])
        .size()
        .unstack(0)
        .fillna(0)
        .astype(int)
    )

## 6. Variación dentro de un tema (puente con el 08)

Si el tema sigue pero cambian sus palabras, el cambio es de **cómo se nombra**. Dentro de los documentos de El Tiempo (base) cuyo tópico dominante es k con θ ≥ 0,4, se comparan períodos con **log-odds con prior informativo de Dirichlet** (Monroe et al. 2008; prior = vocabulario del tópico, α₀ = 500). Se compara con el mismo cálculo sin condicionar al tópico. Además, la **JS con mezcla de tópicos fija** dice cuánto del cambio de vocabulario entre períodos queda cuando se elimina la composición temática. Tokens: los del 08 (A+LB).

In [ ]:
with cr.cronometro(TIEMPOS, "logodds"):
    m = SERIES["eltiempo_base"] & dfp["topico"].notna().values
    puro = m & (dfp["theta_dom"].fillna(1).values >= cfg.theta_min)
    TOKA = {
        i: agenda.reconstruir_tokens(
            dfp.at[i, "lemas_contenido"],
            dfp.at[i, "pos_contenido"],
            dfp.at[i, "cortes_contenido"],
            bigramas=LB,
            renombres=RENOM,
            mapa=MAPA,
        )
        for i in np.where(m)[0]
    }
    cont, cont_all, ndoc = defaultdict(Counter), defaultdict(Counter), Counter()
    for i in np.where(m)[0]:
        cont_all[per[i]].update(TOKA[i])
        if puro[i]:
            k = dfp.at[i, "topico"]
            cont[(k, per[i])].update(TOKA[i])
            ndoc[(k, per[i])] += 1
    PARES = [("P1", "P4"), ("P1", "P2"), ("P2", "P3"), ("P3", "P4")]
    topicos = [t for t in sorted(NOMBRE) if t not in RESIDUALES]
    prior_all = sum(cont_all.values(), Counter())
    filas, comp = [], []
    for a, b in PARES:
        un = cr.logodds_dirichlet(
            cont_all[a], cont_all[b], prior_all, cfg.a0, cfg.min_total_logodds
        )
        for lado, s in [
            (a, un.head(cfg.top_logodds)),
            (b, un.tail(cfg.top_logodds)[::-1]),
        ]:
            for r, x in enumerate(s.itertuples(), 1):
                filas.append(
                    {
                        "topico": "TODOS",
                        "nombre": "sin condicionar",
                        "par": f"{a}-{b}",
                        "lado": lado,
                        "rango": r,
                        "termino": x.termino,
                        "z": x.z,
                        "n_i": x.n_i,
                        "n_j": x.n_j,
                    }
                )
        top_un = set(un.head(50).termino) | set(un.tail(50).termino)
        for k in topicos:
            ci, cj = cont.get((k, a), Counter()), cont.get((k, b), Counter())
            if (
                sum(ci.values()) < cfg.min_tokens_grupo // 4
                or sum(cj.values()) < cfg.min_tokens_grupo // 4
            ):
                comp.append(
                    {
                        "topico": k,
                        "par": f"{a}-{b}",
                        "docs_a": ndoc[(k, a)],
                        "docs_b": ndoc[(k, b)],
                        "estado": "datos_insuficientes",
                    }
                )
                continue
            prior = ci + cj
            lo = cr.logodds_dirichlet(ci, cj, prior, cfg.a0, cfg.min_total_logodds)
            for lado, s in [
                (a, lo.head(cfg.top_logodds)),
                (b, lo.tail(cfg.top_logodds)[::-1]),
            ]:
                for r, x in enumerate(s.itertuples(), 1):
                    filas.append(
                        {
                            "topico": k,
                            "nombre": NOMBRE[k],
                            "par": f"{a}-{b}",
                            "lado": lado,
                            "rango": r,
                            "termino": x.termino,
                            "z": x.z,
                            "n_i": x.n_i,
                            "n_j": x.n_j,
                        }
                    )
            top_k = set(lo.head(50).termino) | set(lo.tail(50).termino)
            comp.append(
                {
                    "topico": k,
                    "par": f"{a}-{b}",
                    "docs_a": ndoc[(k, a)],
                    "docs_b": ndoc[(k, b)],
                    "estado": "ok",
                    "jaccard_top100_vs_sin_condicionar": len(top_k & top_un)
                    / len(top_k | top_un),
                    "z_abs_p95": float(np.quantile(np.abs(lo.z), 0.95)),
                }
            )
LOD = pd.DataFrame(filas)
LOD.to_csv(RES / "logodds_por_topico.csv", index=False)
COMP = pd.DataFrame(comp)
print(
    COMP[COMP.par == "P1-P4"]
    .assign(nombre=lambda x: x.topico.map(NOMBRE))
    .round(3)
    .to_string(index=False)
)
for k in [t for t in cfg.topicos_foco if t in topicos]:
    s = LOD[(LOD.topico == k) & (LOD.par == "P1-P4")]
    print(
        f"\n{k} {NOMBRE[k]} · P1:",
        " ".join(s[s.lado == "P1"].termino),
        "\n   P4:",
        " ".join(s[s.lado == "P4"].termino),
    )
s = LOD[(LOD.topico == "TODOS") & (LOD.par == "P1-P4")]
print(
    "\nSin condicionar · P1:",
    " ".join(s[s.lado == "P1"].termino),
    "\n   P4:",
    " ".join(s[s.lado == "P4"].termino),
)

In [ ]:
pi = pd.Series({k: sum(ndoc[(k, p)] for p, *_ in cfg.periodos) for k in topicos})
pi = pi / pi.sum()
vocab_js = [w for w, _ in prior_all.most_common(20000)]
JS = pd.DataFrame(
    [
        {"par": f"{a}-{b}", **cr.js_estandarizado(cont, a, b, pi, vocab_js)}
        for a, b in PARES
    ]
)
JS.to_csv(RES / "js_mezcla_fija.csv", index=False)
print(
    "JS entre períodos con la mezcla real de tópicos y con la mezcla fija:\n",
    JS.round(4).to_string(index=False),
)

## 7. La paz, medida léxicamente

El tópico de paz del 08 (T10) quedó residual por ser un registro de opinión que depende de la longitud, así que la cobertura de la paz se mide aquí con el léxico: pmw por fuente y año con IC (bootstrap de Poisson por documento), frecuencia documental por mil documentos, y en qué tópicos del 08 caen sus apariciones. Las denominaciones ("proceso de paz", "diálogos de paz"…) están en el conjunto congelado `paz_denominacion` (sección 3) y los vecinos de paz, diálogo, negociación y acuerdo_paz en la sección 4.

In [ ]:
with cr.cronometro(TIEMPOS, "paz"):
    CP = cr.contar_independientes(TXT, TERMINOS_PAZ)
    filas = []
    for s in SERIES_VAR:
        for y in sorted(set(año[SERIES[s]])):
            idx = np.where(SERIES[s] & (año == y))[0]
            Ntok = L_forma[idx].sum()
            Xg = CP.reindex(idx, fill_value=0).values
            ok = Ntok >= cfg.min_tokens_grupo
            lo, hi, _ = (
                cr.bootstrap_tasa(Xg, L_forma[idx], cfg.b_boot, cfg.seed)
                if ok
                else (np.full(Xg.shape[1], np.nan),) * 2 + (None,)
            )
            pm = Xg.sum(0) / Ntok * 1e6
            for k, t in enumerate(CP.columns):
                filas.append(
                    {
                        "serie": s,
                        "año": y,
                        "termino": t,
                        "n": int(Xg[:, k].sum()),
                        "docs": len(idx),
                        "tokens": int(Ntok),
                        "pmw": pm[k] if ok else np.nan,
                        "lo": lo[k],
                        "hi": hi[k],
                        "df_por_mil": (Xg[:, k] > 0).mean() * 1000 if ok else np.nan,
                    }
                )
PAZ = pd.DataFrame(filas)
PAZ.to_csv(RES / "paz_lexica.csv", index=False)
print(
    PAZ[PAZ.serie == "eltiempo_base"]
    .pivot(index="año", columns="termino", values="pmw")
    .round(1)
    .to_string()
)
# ¿En qué tópicos del 08 caen las apariciones de "paz"? (El Tiempo base)
ip = CP.index[CP["paz"] > 0]
dd = pd.DataFrame(
    {
        "n": CP.loc[ip, "paz"].values,
        "topico": dfp.loc[ip, "topico"].values,
        "periodo": per[ip],
        "m": SERIES["eltiempo_base"][ip],
    }
)
dd = dd[dd.m & dd.periodo.notna()]
dd["grupo"] = dd.topico.where(
    dd.topico.isin(["T10", "T02", "T08", "T13", "T16"]), "otros"
)
print(
    "\nReparto de las apariciones de 'paz' por tópico dominante del 08:\n",
    dd.groupby(["periodo", "grupo"])["n"]
    .sum()
    .unstack()
    .pipe(lambda x: x.div(x.sum(1), axis=0))
    .round(3)
    .to_string(),
)
fig, axs = plt.subplots(1, 2, figsize=(14, 3.8))
for ax, t in zip(axs, ["paz", "proceso de paz"]):
    for s, c in [
        ("eltiempo_base", "C0"),
        ("semana_base", "C1"),
        ("eltiempo_todos", "C2"),
    ]:
        g = PAZ[(PAZ.serie == s) & (PAZ.termino == t)].sort_values("año")
        ax.plot(g.año, g.pmw, c=c, label=s)
        ax.fill_between(g.año, g.lo, g.hi, color=c, alpha=0.2)
    ax.axvline(cfg.año_formato, c="grey", lw=0.8)
    ax.set_title(f"pmw de '{t}' con IC 95 %")
    ax.legend(fontsize=7)
guardar(fig, "fig_paz_lexica")

## 8. EXPLORATORIO: descubrimiento de reemplazos

**Esta sección no es confirmatoria.** Se buscan pares "misma referencia, otra palabra" que no estaban en la lista congelada: para cada lema que cae a menos de la mitad entre dos períodos consecutivos (log₂ ≤ −1, pmw ≥ 5 antes), su vecino más cercano en el espacio del período siguiente (alineado con Procrustes) entre los lemas que suben al doble o más. Solo pares que aparecen en el top-3 de ≥ 3 de 5 semillas, sin entidades. Se revisan a mano con contextos. Los conjuntos nuevos que salgan de aquí se registran en `EXPLORATORIO` (fuera del hash) y no se mezclan con la sección 3.

In [ ]:
EXPLORATORIO = {}  # conjuntos nuevos descubiertos aquí (no entran al hash congelado)
# Parámetros exploratorios (fuera del hash). Primera corrida con pmw ≥ 5 y modelos de semillas: solo ruido de
# palabras raras (cocina, moda; sim ≈ 0,9 = región de palabras raras). Se sube a pmw ≥ 20 en ambos períodos
# y se usan las réplicas bootstrap de la 4b con su vocabulario.
REEMP_MIN_PMW = 20.0
filas = []
ps = sorted(MODELOS_B["base"])
for a, b in zip(ps[:-1], ps[1:]):
    r = cr.reemplazos_candidatos(
        MODELOS_B["base"][a],
        MODELOS_B["base"][b],
        VOC_B,
        FREC["base"][a],
        FREC["base"][b],
        cfg,
        excluir=PROPIOS_LEMA,
        min_pmw=REEMP_MIN_PMW,
    )
    if len(r):
        filas.append(r.assign(transicion=f"{a}→{b}"))
REEMP = pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()
if len(REEMP):
    TXTL = {}

    def ctx_lema(w, p, n=2):
        pos = [i for i in np.where(SERIES["eltiempo_base"] & (per == p))[0]]
        rx, mp_ = cr.compilar({w: [w]})
        for i in pos:
            if i not in TXTL:
                TXTL[i] = " " + " ".join(TOKL[i]) + " "
        return cr.contextos(
            TXTL,
            ids,
            [i for i in pos if f" {w} " in TXTL[i]],
            rx,
            mp_,
            w,
            n,
            cfg.ventana_kwic,
            cfg.seed,
        )

    top = REEMP.groupby("transicion").head(12).copy()
    top["ctx_cae"] = [
        " | ".join(ctx_lema(w, t.split("→")[0]))
        for w, t in zip(top.cae, top.transicion)
    ]
    top["ctx_candidato"] = [
        " | ".join(ctx_lema(v, t.split("→")[1]))
        for v, t in zip(top.candidato, top.transicion)
    ]
    REEMP = REEMP.merge(
        top[["cae", "candidato", "transicion", "ctx_cae", "ctx_candidato"]], how="left"
    )
    print(
        REEMP.groupby("transicion")
        .head(12)[
            [
                "transicion",
                "cae",
                "candidato",
                "n_semillas",
                "sim_media",
                "pmw_cae_antes",
                "pmw_cae_despues",
                "pmw_cand_antes",
                "pmw_cand_despues",
            ]
        ]
        .round(2)
        .to_string(index=False)
    )
else:
    print(
        "Sin candidatos que cumplan los filtros (resultado explícito, no se relaja la regla aquí)."
    )
REEMP.to_csv(RES / "reemplazos_candidatos.csv", index=False)

### 8b. EXPLORATORIO: regímenes de codificación del archivo

Descubierto en la primera corrida: "vergenza" (hasta 1999) → "verguenza" (2000–2004) → "vergüenza" (desde 2006); "antigedad", "arbitro"; y el pmw de *sólo* y *éste* cae a la mitad en El Tiempo 2006–2008 mientras Semana sigue estable hasta 2010. Hipótesis: el archivo de El Tiempo trata tildes y diéresis distinto según el régimen de digitalización. Se mide con:
* palabras cuya forma sin tilde **no** es otra palabra válida (también/tambien, país/pais, política/politica…): cuota de la forma sin tilde;
* palabras con ü: cuota con ü / con u / con la letra eliminada;
* % de tokens con vocal tildada.
Si hay regímenes, los resultados **ortográficos** de El Tiempo (sólo, período, guion) quedan confundidos con el archivo y solo Semana sirve para ortografía.

In [ ]:
CON_TILDE = [
    "también",
    "después",
    "según",
    "además",
    "país",
    "países",
    "política",
    "políticas",
    "político",
    "económica",
    "económico",
    "región",
    "nación",
    "acción",
    "información",
    "educación",
    "situación",
    "público",
    "días",
    "último",
]
CON_TILDE = [
    w
    for w in CON_TILDE
    if w.translate(str.maketrans("áéíóú", "aeiou")) not in ("publico", "ultimo", "dias")
]  # válidas sin tilde
CON_DIERESIS = [
    "vergüenza",
    "antigüedad",
    "bilingüe",
    "bilingüismo",
    "lingüística",
    "lingüístico",
    "pingüino",
    "ambigüedad",
    "desagüe",
    "agüero",
    "cigüeña",
    "nicaragüense",
    "nicaragüenses",
    "halagüeño",
    "argüir",
]
PARES = cr.pares_diacriticos(CON_TILDE, CON_DIERESIS)
etq_cod = [(CORTO[f], int(a)) for f, a in zip(fuente, año)]
CF_COD = cr.cuotas_formas(Xf, ixf, etq_cod, PARES)
CF_COD[["fuente", "año"]] = pd.DataFrame(CF_COD["grupo"].tolist(), index=CF_COD.index)
til = (
    CF_COD[CF_COD.conjunto.isin(CON_TILDE)]
    .groupby(["fuente", "año", "variante"])["n"]
    .sum()
    .unstack()
)
til["pct_sin_tilde"] = til["sin_tilde"] / til.sum(1) * 100
die = (
    CF_COD[CF_COD.conjunto.isin(CON_DIERESIS)]
    .groupby(["fuente", "año", "variante"])["n"]
    .sum()
    .unstack()
)
die["n_total"] = die.sum(1)
for v in ["ü", "u", "eliminada"]:
    die[f"pct_{v}"] = die[v] / die["n_total"] * 100
TD = cr.tasa_diacriticos(Xf, vf, L_forma, etq_cod)
TD[["fuente", "año"]] = pd.DataFrame(TD["grupo"].tolist(), index=TD.index)
COD = (
    til[["pct_sin_tilde"]]
    .join(die[["n_total", "pct_ü", "pct_u", "pct_eliminada"]])
    .reset_index()
    .merge(TD.drop(columns="grupo"), on=["fuente", "año"])
)
COD.to_csv(RES / "diagnostico_codificacion.csv", index=False)
print(
    COD[COD.fuente.isin(["eltiempo", "semana"])]
    .pivot(
        index="año",
        columns="fuente",
        values=["pct_sin_tilde", "pct_tilde", "pct_ü", "pct_eliminada"],
    )
    .round(2)
    .to_string()
)
for f in ["eltiempo", "semana"]:
    g = COD[COD.fuente == f].set_index("año").sort_index()
    for col in ["pct_sin_tilde", "pct_tilde", "pct_eliminada"]:
        x = g[col].dropna()
        if len(x) >= 2 * cfg.min_seg + 1:
            cps = agenda.binseg(
                ((x - x.mean()) / (x.std() + 1e-12)).values[:, None],
                np.ones(len(x)),
                cfg.min_seg,
                cfg.b_binseg,
                cfg.q_binseg,
                seed=cfg.seed,
            )
            print(f"{f:9s} {col:15s} cambios: {[int(x.index[i]) for i, _ in cps]}")
# ¿Explica la tasa de tildes la caída de 'sólo' y 'éste' en El Tiempo?
t_et = TILDES[TILDES.serie == "eltiempo_base"].pivot(
    index="año", columns="termino", values="pmw"
)
c_et = COD[COD.fuente == "eltiempo"].set_index("año")["pct_tilde"]
print(
    "\nSpearman por año (El Tiempo) entre % tokens con tilde y pmw:",
    {
        t: round(
            stats.spearmanr(
                t_et[t], c_et.reindex(t_et.index), nan_policy="omit"
            ).statistic,
            3,
        )
        for t in t_et.columns
    },
)
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for f, g in COD[COD.fuente.isin(["eltiempo", "semana"])].groupby("fuente"):
    axs[0].plot(g.año, g.pct_sin_tilde, marker="o", ms=3, label=f)
    axs[1].plot(g.año, g.pct_tilde, marker="o", ms=3, label=f)
    axs[2].plot(g.año, g.pct_eliminada, marker="o", ms=3, label=f"{f} ü eliminada")
    axs[2].plot(g.año, g.pct_u, ls=":", label=f"{f} ü→u")
axs[0].set_title("% sin tilde (palabras que la exigen)")
axs[1].set_title("% de tokens con vocal tildada")
axs[2].set_title("palabras con ü")
for a in axs:
    a.axvline(cfg.año_formato, c="grey", lw=0.8)
    a.axvline(2000, c="grey", ls=":", lw=0.8)
    a.legend(fontsize=7)
guardar(fig, "fig_diagnostico_codificacion")

## 9. Resumen y exportes

**Notas para el informe (sin cambios en S0):** spaCy md lematiza "alias" como "alia" (aparece en los log-odds de T02); se registra como pendiente de S0 junto a los ya conocidos y no se corrige aquí porque exigiría volver a correr el 07 y el 08.

In [ ]:
cru_et = CRU[CRU.serie == "eltiempo_base"]
RESUMEN = {
    "hash_congelado": HASH,
    "fast": FAST,
    "config": cfg.a_dict(),
    "n_docs": int(len(dfp)),
    "series_docs": {k: int(v.sum()) for k, v in SERIES.items()},
    "rareza": {
        "piso_zipf_min_max": {
            f: [round(g.piso_zipf.min(), 2), round(g.piso_zipf.max(), 2)]
            for f, g in NG.groupby("fuente")
        },
        "grupos_sin_banda_rara_medible": NG.loc[~NG.rara_medible, ["fuente", "año"]]
        .astype(str)
        .agg(" ".join, axis=1)
        .tolist(),
        "conteo_minimo_eltiempo": CONTEO_MIN,
        "cociente_ic_boot_poisson_mediana": float(AR["cociente"].median()),
        "cociente_ic_boot_poisson_mediana_n10": float(AR10["cociente"].median()),
        "spearman_cociente_dp": float(
            stats.spearmanr(AR["cociente"], AR["dp"], nan_policy="omit").statistic
        ),
        "spearman_pmw_dfpormil": float(
            stats.spearmanr(DD.pmw, DD.df_por_mil).statistic
        ),
        "desplazamiento_zipf_denominador": den.groupby("fuente")[
            "desplazamiento_zipf_base"
        ]
        .agg(["min", "max"])
        .round(3)
        .to_dict("index"),
        "frac_propias_top200": float(
            DISP[DISP.contenido].nlargest(200, "pmw").propio.mean()
        ),
    },
    "variantes": cru_et[
        ["conjunto", "tipo", "estado", "cruce", "lo", "hi", "cerca_formato", "cambios"]
    ].to_dict("records"),
    "lectura_controles": LECT.reset_index().to_dict("records"),
    "semantico": {
        "umbral_vocab": U,
        "n_vocab": len(VOC),
        "control_frecuencia": CF,
        "cuentan_por_transicion": DESP.groupby("transicion")["cuenta"].sum().to_dict(),
    },
    "neologismos": {
        f"{t}|{'entidad' if e else 'comun'}": int(n)
        for (t, e), n in NEO[NEO.umbral == cfg.umbral_neo]
        .groupby(["tipo", "entidad"])
        .size()
        .items()
    },
    "js_mezcla_fija": JS.to_dict("records"),
    "semantico_bootstrap": {
        "umbral_vocab": U_B,
        "n_vocab": len(VOC_B),
        "control_frecuencia": CF_B,
        "comparacion_con_semillas": cmp4.round(4).to_dict("index"),
        "cuentan_por_transicion": DESP_B.groupby("transicion")["cuenta"]
        .sum()
        .to_dict(),
    },
    "codificacion": {
        f: {
            c: [round(float(g[c].min()), 3), round(float(g[c].max()), 3)]
            for c in ["pct_sin_tilde", "pct_tilde", "pct_eliminada"]
        }
        for f, g in COD.groupby("fuente")
    },
    "reemplazos_candidatos": int(len(REEMP)),
    "reemplazos_min_pmw": REEMP_MIN_PMW,
    "notas_informe": [
        "spaCy md lematiza 'alias' como 'alia' (pendiente de S0, no corregido)",
        "sección 4b: control de ruido corregido post hoc (bootstrap de documentos); la sección 4 se conserva",
    ],
    "tiempos_s": TIEMPOS,
    "total_min": round((time.time() - T0) / 60, 1),
}
agenda.a_json(RESUMEN, RES / "resumen_09.json")
print(
    json.dumps(
        {k: RESUMEN[k] for k in ["hash_congelado", "tiempos_s", "total_min"]},
        indent=1,
        ensure_ascii=False,
    )
)
print(sorted(p.name for p in RES.iterdir()))